# Code Classification with BERT Embeddings

In [2]:
from transformers import BertTokenizer, BertModel
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

In [3]:
# Initialize tokenizer + model once
print('Loading BERT tokenizer and model...')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Function to get embeddings for text samples
def get_bert_embeddings(text, max_length=512):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=max_length)
    with torch.no_grad():
        outputs = model(**inputs)
    # Use [CLS] pooled representation
    embeddings = outputs.last_hidden_state[:, 0, :].numpy()
    return embeddings.flatten()

Loading BERT tokenizer and model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Load the feather data file
df = pd.read_feather('cleaned_data.feather')

# Embed text to BERT vectors in batch and save for training
# (Tokenizer and model already initialized at the top cell.)

def get_bert_embeddings_batch(texts, batch_size=64, max_length=512, device='cpu'):
    model.to(device)
    all_emb = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Bert batches'):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=max_length
        ).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_emb.append(emb)
    return np.vstack(all_emb)

print("Generating BERT embeddings in batches...")
texts = df['content'].astype(str).tolist()
X = get_bert_embeddings_batch(texts, batch_size=64, device='cuda' if torch.cuda.is_available() else 'cpu')
y = df['language'].values

In [ ]:
# X and y to dataframe
df_embeddings = pd.DataFrame(X)
df_embeddings['language'] = y
#rename features to X1, X2, ...
df_embeddings.rename(columns={i: f'X{i+1}' for i in range(X.shape[1])}, inplace=True)
#This is the file provided to you for training the classifier. You can load it with pd.read_feather('embeddings.feather')
df_embeddings.to_feather('embeddings.feather')

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,...,X760,X761,X762,X763,X764,X765,X766,X767,X768,language
0,-0.173793,-0.325611,0.260916,-0.090371,-0.003099,-0.689703,0.169172,0.161007,-0.156974,-0.634382,...,-0.444923,-0.085975,-0.347440,-0.071519,0.916738,0.032165,-0.395401,-0.151372,0.753215,Markdown
1,-0.551476,0.114461,-0.156257,-0.095483,-0.491529,-0.717104,0.610803,-0.087726,0.567443,-0.527859,...,-0.543031,-0.006777,-0.379004,-0.076810,0.353173,-0.265057,-0.592026,-0.525372,0.525150,XML
2,-0.340610,0.312390,-0.245592,-0.390323,0.570317,-0.937024,0.177742,1.024876,0.044157,0.052844,...,-0.521548,0.697367,-0.445880,-0.071272,0.251338,0.437460,-0.264365,-0.438744,0.614954,Text
3,0.030533,-0.629911,-0.322072,0.253435,-0.335275,-0.971226,0.205228,0.110962,0.084097,-0.191846,...,-0.457321,0.029477,-0.645487,-0.347078,1.028343,-0.073201,-0.468006,-0.304346,0.600119,Markdown
4,-0.077602,-0.384216,-0.106897,0.131760,0.150618,-0.788030,0.596326,0.423850,0.306056,-0.540935,...,-0.061873,-0.118206,-0.583353,-0.400244,0.484490,-0.250504,-0.217052,-0.076143,0.817761,Markdown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86189,-0.323911,-0.439272,-0.170922,0.068886,0.125510,-0.256554,0.536814,0.348785,-0.360812,-0.175219,...,-0.343193,0.362776,-0.271427,0.194223,0.888953,-0.179987,-0.430233,-0.061140,0.831387,SQL
86190,-0.410431,-0.221323,-0.050184,-0.214193,-0.141798,-0.409090,-0.030539,0.126609,0.169301,0.114508,...,-0.599389,0.373161,-0.137648,0.017733,0.681858,-0.473497,-0.298967,-0.028612,1.083044,SQL
86191,-0.323911,-0.439272,-0.170922,0.068886,0.125510,-0.256554,0.536814,0.348785,-0.360812,-0.175219,...,-0.343193,0.362776,-0.271427,0.194223,0.888953,-0.179987,-0.430233,-0.061140,0.831387,SQL
86192,-0.410431,-0.221323,-0.050184,-0.214193,-0.141798,-0.409090,-0.030539,0.126609,0.169301,0.114508,...,-0.599389,0.373161,-0.137648,0.017733,0.681858,-0.473497,-0.298967,-0.028612,1.083044,SQL


In [7]:
df_embeddings = pd.read_feather('embeddings.feather')
df_embeddings.head()

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,...,X760,X761,X762,X763,X764,X765,X766,X767,X768,language
0,-0.173793,-0.325611,0.260916,-0.090371,-0.003099,-0.689703,0.169172,0.161007,-0.156974,-0.634382,...,-0.444923,-0.085975,-0.347440,-0.071519,0.916738,0.032165,-0.395401,-0.151372,0.753215,Markdown
1,-0.551476,0.114461,-0.156257,-0.095483,-0.491529,-0.717104,0.610803,-0.087726,0.567443,-0.527859,...,-0.543031,-0.006777,-0.379004,-0.076810,0.353173,-0.265057,-0.592026,-0.525372,0.525150,XML
2,-0.340610,0.312390,-0.245592,-0.390323,0.570317,-0.937024,0.177742,1.024876,0.044157,0.052844,...,-0.521548,0.697367,-0.445880,-0.071272,0.251338,0.437460,-0.264365,-0.438744,0.614954,Text
3,0.030533,-0.629911,-0.322072,0.253435,-0.335275,-0.971226,0.205228,0.110962,0.084097,-0.191846,...,-0.457321,0.029477,-0.645487,-0.347078,1.028343,-0.073201,-0.468006,-0.304346,0.600119,Markdown
4,-0.077602,-0.384216,-0.106897,0.131760,0.150618,-0.788030,0.596326,0.423850,0.306056,-0.540935,...,-0.061873,-0.118206,-0.583353,-0.400244,0.484490,-0.250504,-0.217052,-0.076143,0.817761,Markdown
